# 노이즈-인지 앙상블 + bootstrap 95% CI — Mahalanobis와 통계적 동급인지 판정


In [ ]:
import os, subprocess, pickle
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
SEED=42
def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))
HN=(_find('hard_negative_features_v2.pkl') or [None])[0]
assert HN, 'hard_negative_features_v2.pkl 필요'
DATA=pickle.load(open(HN,'rb')); print('loaded', list(DATA.keys()))
FEATS=['hfe','gl','predl1']
def arr(rows,k): return np.array([r[k] for r in rows],float)
def split(rows,seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(len(rows)); rng.shuffle(idx); h=len(idx)//2
    return [rows[i] for i in idx[:h]],[rows[i] for i in idx[h:]]

def auc_ci(neg,pos,B=2000,seed=SEED):
    y=np.r_[np.zeros(len(neg)),np.ones(len(pos))]; s=np.r_[neg,pos]
    base=roc_auc_score(y,s); rng=np.random.RandomState(seed); b=[]
    neg=np.asarray(neg); pos=np.asarray(pos)
    for _ in range(B):
        nb=neg[rng.randint(0,len(neg),len(neg))]; pb=pos[rng.randint(0,len(pos),len(pos))]
        yy=np.r_[np.zeros(len(nb)),np.ones(len(pb))]; ss=np.r_[nb,pb]
        b.append(roc_auc_score(yy,ss))
    lo,hi=np.percentile(b,[2.5,97.5]); return base,lo,hi
print('ready')

In [ ]:
REF_MAHA={'CIFAR-10':0.8855,'ImageNet_eps8':None}   # 표9 +하드네거 기준선
def aggregators(cal_c,cal_h,cal_a):
    def Zmk(mu,sd):
        return lambda rows: np.stack([(arr(rows,k)-mu[k])/sd[k] for k in FEATS],1)
    mu={k:arr(cal_c,k).mean() for k in FEATS}; sd={k:arr(cal_c,k).std()+1e-8 for k in FEATS}
    Z=Zmk(mu,sd)
    lr=LogisticRegression(max_iter=2000,class_weight='balanced').fit(
        np.concatenate([Z(cal_c),Z(cal_h),Z(cal_a)],0),
        np.r_[np.zeros(len(cal_c)+len(cal_h)),np.ones(len(cal_a))])
    aggs={'mean(baseline)':lambda Zm:Zm.mean(1),
          'pred-only':lambda Zm:Zm[:,[1,2]].mean(1),
          'median':lambda Zm:np.median(Zm,1),
          'top2':lambda Zm:np.sort(Zm,1)[:,-2:].mean(1),
          'logistic*':lambda Zm:lr.predict_proba(Zm)[:,1]}
    return Z,aggs,lr.coef_[0]

for ds,d in DATA.items():
    clean,adv,hard=d['clean'],d['adv'],d['hard']; hardrows=[r for t in hard for r in hard[t]]
    cal_c,te_c=split(clean); cal_a,te_a=split(adv); cal_h,te_h=split(hardrows)
    Z,aggs,w=aggregators(cal_c,cal_h,cal_a)
    Zc,Za,Zh=Z(te_c),Z(te_a),Z(te_h)
    ref=REF_MAHA.get(ds)
    print(f"\n[{ds}]  Mahalanobis +하드네거 기준선 = {ref}")
    print(f"  {'집계기':<16}{'pristine AUC[95%CI]':>26}{'+하드네거 AUC[95%CI]':>28}{'vs Maha':>12}")
    for name,f in aggs.items():
        bp,lp,hp=auc_ci(f(Zc),f(Za))
        bh,lh,hh=auc_ci(np.r_[f(Zc),f(Zh)],f(Za))
        verdict=''
        if ref is not None:
            if lh<=ref<=hh: verdict='동급(겹침)'
            elif bh>ref: verdict='상회'
            else: verdict='하회'
        print(f"  {name:<16}{f'{bp:.3f}[{lp:.3f},{hp:.3f}]':>26}{f'{bh:.3f}[{lh:.3f},{hh:.3f}]':>28}{verdict:>12}")
    print(f"  logistic 가중치 [zHF,zGL,zPL]=[{w[0]:+.2f},{w[1]:+.2f},{w[2]:+.2f}]")
print("\n판정 규칙: Maha 점추정(0.8855)이 집계기의 +하드네거 95%CI 안에 들면 '동급'.")
print("(엄밀한 paired 검정은 동일 분할에서 Maha 표본별 점수가 필요 — SOTA 노트북에서 점수 저장 시 가능.)")